# 1. Import Libraries

In [1]:
!pip install transformers datasets 'accelerate>=1.1.0'
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
import torch


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


/home/lenovo/Documents/School/Semester 6/PBA/kelompok-7-pba/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Load Model

In [2]:
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3207.80it/s]


# 3. Prepare Dataset

In [12]:
df = pd.read_csv("../dataset/dataset.csv")

labels_map = {
    "Positive": 2,
    "Netral": 1,
    "Negative": 0
}

df["manual_sentiment_mapped"] = df["Manual Sentiment"].map(labels_map)
df

,URL,Judul,Konten,Manual Sentiment,manual_sentiment_mapped
0,https://www.cnbcindonesia.com/research/2025012...,Trump Sebar Exceutive Order: Emang Semengerika...,"Jakarta, CNBC Indonesia -Amerika Serikat (AS) ...",Positive,2
1,https://www.cnbcindonesia.com/research/2025012...,"Alasan Rupiah 'Berpesta' di Pelantikan Trump, ...","Jakarta, CNBC Indonesia -Nilai tukar rupiah te...",Positive,2
2,https://www.cnbcindonesia.com/research/2025012...,"Trump Beri Kabar Baik, Saatnya Menunggu Dolar ...","Jakarta, CNBC Indonesia-Pasar keuangan Indones...",Positive,2
3,https://www.cnbcindonesia.com/research/2025030...,"IHSG Merah Lagi, Begini Penjelasan dari Analis...","Jakarta, CNBC Indonesia -Indeks Harga Saham Ga...",Negative,0
4,https://indodax.com/academy/bitcoin-200k-predi...,Bernstein: Bitcoin Bisa Naik 2x Lipat! Target ...,HargaBitcoin(BTC)pernah melewati angka terting...,Positive,2
...,...,...,...,...,...
996,https://voi.id/teknologi/489452/produsen-mesin...,Produsen Mesin Bitcoin China Pindahkan Produks...,JAKARTA - Tiga produsen mesin penambang bitcoi...,Positive,2
997,https://www.liputan6.com/bisnis/read/6058462/d...,Donald Trump Kembali Tekan Ketua The Fed Jerom...,Presiden Amerika Serikat (AS) Donald Trump men...,Netral,1
998,https://kabar24.bisnis.com/read/20250622/19/18...,"Donald Trump Serang Iran, Ekonomi Global Bakal...","Bisnis.com,JAKARTA — Serangan Amerika Serikat ...",Negative,0
999,https://www.cnbcindonesia.com/tech/20250620153...,Trump 'Palak' Raksasa Teknologi Rp 900 Triliun...,"Jakarta, CNBC Indonesia -Perusahaan semikonduk...",Negative,0


# 4. Tokenise Data

In [14]:
encodings = tokenizer(
    df["Konten"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128
)

df["input_ids"] = encodings["input_ids"]
df["attention_mask"] = encodings["attention_mask"]
df

,URL,Judul,Konten,Manual Sentiment,manual_sentiment_mapped,input_ids,attention_mask
0,https://www.cnbcindonesia.com/research/2025012...,Trump Sebar Exceutive Order: Emang Semengerika...,"Jakarta, CNBC Indonesia -Amerika Serikat (AS) ...",Positive,2,"[0, 6758, 16, 35210, 7778, 622, 824, 26975, 44...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,https://www.cnbcindonesia.com/research/2025012...,"Alasan Rupiah 'Berpesta' di Pelantikan Trump, ...","Jakarta, CNBC Indonesia -Nilai tukar rupiah te...",Positive,2,"[0, 6758, 16, 35210, 7778, 622, 824, 20028, 11...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,https://www.cnbcindonesia.com/research/2025012...,"Trump Beri Kabar Baik, Saatnya Menunggu Dolar ...","Jakarta, CNBC Indonesia-Pasar keuangan Indones...",Positive,2,"[0, 6758, 16, 35210, 7778, 40175, 7753, 17, 21...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,https://www.cnbcindonesia.com/research/2025030...,"IHSG Merah Lagi, Begini Penjelasan dari Analis...","Jakarta, CNBC Indonesia -Indeks Harga Saham Ga...",Negative,0,"[0, 6758, 16, 35210, 7778, 622, 824, 36568, 29...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,https://indodax.com/academy/bitcoin-200k-predi...,Bernstein: Bitcoin Bisa Naik 2x Lipat! Target ...,HargaBitcoin(BTC)pernah melewati angka terting...,Positive,2,"[0, 5620, 34950, 12, 38, 9786, 13, 38933, 3888...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
...,...,...,...,...,...,...,...
996,https://voi.id/teknologi/489452/produsen-mesin...,Produsen Mesin Bitcoin China Pindahkan Produks...,JAKARTA - Tiga produsen mesin penambang bitcoi...,Positive,2,"[0, 11178, 824, 10177, 5159, 1740, 23062, 7542...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
997,https://www.liputan6.com/bisnis/read/6058462/d...,Donald Trump Kembali Tekan Ketua The Fed Jerom...,Presiden Amerika Serikat (AS) Donald Trump men...,Netral,1,"[0, 14312, 2502, 4439, 423, 2544, 13, 19359, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
998,https://kabar24.bisnis.com/read/20250622/19/18...,"Donald Trump Serang Iran, Ekonomi Global Bakal...","Bisnis.com,JAKARTA — Serangan Amerika Serikat ...",Negative,0,"[0, 11654, 18, 1003, 16, 11178, 7662, 24805, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
999,https://www.cnbcindonesia.com/tech/20250620153...,Trump 'Palak' Raksasa Teknologi Rp 900 Triliun...,"Jakarta, CNBC Indonesia -Perusahaan semikonduk...",Negative,0,"[0, 6758, 16, 35210, 7778, 622, 824, 12295, 45...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


# 5. Convert to Torch

In [15]:
input_ids = torch.tensor(encodings["input_ids"])
attention_mask = torch.tensor(encodings["attention_mask"])
labels = torch.tensor(df["manual_sentiment_mapped"].values).long()

train_dataset = [
    {
        "input_ids": input_ids[i],
        "attention_mask": attention_mask[i],
        "labels": labels[i]
    }
    for i in range(len(labels))
]
train_dataset

[{'input_ids': tensor([    0,  6758,    16, 35210,  7778,   622,   824, 26975,  4439,   423,
           2544,    13,   617,  1781,   530,  2982,   729,   287, 10126,  2659,
            468,  7615,   477,    17, 21979,   287, 21860,    18,   203, 14312,
           2502,  4439,   423,  2544,    13, 19359, 11173,  1781, 13738,   403,
           5329,   698,  2534,   423,  2955,    19,    21,    19,  2955,  4552,
            765,   203,  6675,  1128,    16,   395, 30873,    16, 11173,  4202,
           3057,  4140, 17364,  4059,   287,  1218, 15940,   385,   778,   617,
          12094,  6274,   385,  4958,   458, 22776,    18,   203,  7796,  1059,
           1789,  7615, 29285,  4769, 12414,  4140,    16, 21480,    16, 36492,
           4235,    16,   296,  3598, 35185,    18, 12300,   330,  7653,  1542,
           1705, 18389,  2534,    16,   953,   406, 11376,  6771, 11668,   287,
            429,  4448,  3983,    16, 15743,    16,   296,  4094,  2512, 13747,
           2534,   423,  80

# 6. Train Model

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

/home/lenovo/Documents/School/Semester 6/PBA/kelompok-7-pba/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
